# 16 - Train a Multi-Head-Update DQN Model Offline

This notebook follows the same offline workflow as `02_train_offline_dqn.ipynb`, but each update step trains the Q-head `HEAD_UPDATES` times for every encoder and backbone step:

1. Load previously collected `Datastore` streams from the Hub.
2. Build a `DataLoader` that samples fixed-length sequences from those streams.
3. Assemble a `Model` from an embedder, a backbone, and an action-value head.
4. Train with `DqnObjective` and save with `push_model_to_hub`.

One composite update is: compute last-layer features once, then run the head `m` times toward the Bellman target. The first `m - 1` head steps detach those features (no encoder/backbone grads) and Polyak only the delayed head. The last head step keeps the feature graph so grads reach the encoder and backbone, then Polyak all three sections. `HEAD_UPDATES = 1` is the same as `02`.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in `09_inference.ipynb`.


In [ ]:
import torch

from mouse_core import AdamW
from mouse_core.data import (
    DataLoader,
    Augmenter,
    Tokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import DqnObjective
from mouse_core.models import Model, Polyak, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import NumericEmbedder
from mouse_core.models.heads import DiscreteActionValueHead


DATASET_ID = "mouse-example-dataset"          # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-multi-head-update-offline"  # Hugging Face model repo for push_model_to_hub
TOKENIZER_ID = "mouse-example-tokenizer-multi-head-update-offline"  # Hugging Face tokenizer repo (separate from MODEL_ID)
MAX_ACTIONS = 4                               # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
SEQUENCE_LENGTH = 512                         # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                # sequences per optimizer step
NUM_CYCLES = 2                               # outer train cycles (print cadence)
TRAIN_STEPS = 50                             # composite updates per cycle (each is HEAD_UPDATES head steps + one encoder/backbone step)
HEAD_UPDATES = 4                              # head optimizer steps per encoder/backbone step (1 = same as 02)
POLYAK_TAU_HEADS = 0.0001                     # delayed Q-head interpolation (0 = frozen, 1 = copy of the online heads)
POLYAK_TAU_ENCODER = 0.01                     # delayed encoder interpolation
POLYAK_TAU_BACKBONE = 0.01                    # delayed backbone interpolation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split='train', force_download=True)

## Data pipeline

`DataLoader` samples contiguous windows up to `sequence_length` (a max) from one or more datastores. Each sequence may be shorter than the max depending on where the window starts in the store.

Pipeline order: `augmenter → tokenizer → pack → embedder`.

| Stage | Role |
| --- | --- |
| **Augmenter** | `dict → dict` (`fields=` value transforms; `seed_field=` for shared draws within a `reseed` generation). Action permute sets `input_vector_field` / `output_vector_field` on `info_q_star` so Q* stays aligned. |
| **Tokenizer** | `dict → StepTokens` (`input_field` / `output_field`; `objective_fields=` is `action` / `reward` / `episode_done` / `task_done`; `grouping_field=`) |

Compose `train_transform = compose(augmenter, tokenizer)`.
`DataLoader(transform=train_transform)` maps each step and packs into a `TokenBatch`.
Live inference in `09_inference.ipynb` uses the tokenizer without the augmenter so chosen actions match the env.


In [ ]:
# Pipeline order: augmenter → tokenizer

augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "input_vector_field": "info_q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "type": "discrete",
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

tokenizer = Tokenizer(
    input_fields=[
        {
            "type": "discrete",
            "input_field": "action",
        },
        {
            "type": "discrete",
            "input_field": "observation",
        },
        {
            "type": "fourier",
            "input_field": "reward",
        },
        {
            "type": "discrete",
            "input_field": "episode_done",
        },
        {
            "type": "learnable",
            "output_field": "value",
            "tokens": 1,
            "head_output": True,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
        },
        {
            "input_field": "reward",
        },
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "task_done",
        },
    ],
    grouping_field="task_index",
)

train_transform = compose(augmenter, tokenizer)

loader = DataLoader(
    stores=stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)


## Build The Model

A Mouse Core `Model` has three main pieces:

- `NumericEmbedder` maps a tokenized `TokenBatch` (modalities keyed by name; add `vocab_size` / `std` here; `fourier` / `continuous` also need `fourier_min` / `fourier_max`) into vectors.
- `Qwen3Backbone` processes those tokens with a transformer backbone. Three arguments are required and describe how it runs on this machine rather than what it is, so they are not saved with the model and `load_model` asks for them again: `train_kernel` for the uncached forward (`"flex"`, block-sparse FlexAttention), `decode_kernel` for cached decode (`"flex"`, paged FlexAttention) and `dtype` for the base weights (`torch.float32` so the whole backbone trains in fp32). `model.to(device)` only moves.
- `DiscreteActionValueHead` predicts one value per discrete action.

The backbone exposes `hidden_dim`, and the embedder and head use that same value so the pieces connect cleanly.

`NumericEmbedder` modality types used here:

- `discrete` for integer IDs such as actions, observations, and episode/task done codes.
- `fourier` for scalar numeric values such as rewards.
- `learnable` for the trailing `value` token (no step field; flagged `head_output: True` so Q is read from it).

`Model(...)` wraps the pieces behind a single forward call that returns predictions, last-layer states, and an optional cache. This notebook uses `model.features(inputs)` (encoder + backbone, no heads) and calls `model.head` for each of the `m` head updates.


In [ ]:
backbone = Qwen3Backbone(
    train_kernel="flex",
    decode_kernel="flex",
    dtype=torch.float32,
    pretrained="Qwen/Qwen3-0.6B",
)

encoder = NumericEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "type": "discrete",
            "field": "action",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "discrete",
            "field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "fourier",
            "field": "reward",
            "std": 0.02,
            "positions": 1,
            "fourier_min": 0.01,
            "fourier_max": 10.0,
        },
        {
            "type": "discrete",
            "field": "episode_done",
            "vocab_size": 3,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "learnable",
            "field": "value",
            "tokens": 1,
            "std": 0.02,
            "positions": 1,
        },
    ],
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(
    encoder=encoder,
    backbone=backbone,
    heads=head,
    action_head="action_value",
    reasoner=None,
    recurrence=None,
).train().to(device)
print(model)


## Training Phase

Each outer cycle runs `TRAIN_STEPS` composite updates via `run_train`. Each composite update is `HEAD_UPDATES` head steps and one encoder/backbone step:

1. `inputs, objective_data = loader.next_batch()` samples ragged step windows (up to `SEQUENCE_LENGTH`).
2. `model.features(inputs)` embeds the `TokenBatch` and runs the backbone without running heads. Those last-layer features stay on the tape for the last inner update.
3. `delayed_model.features(inputs)` under `torch.no_grad()` pools delayed features the same way. Those stay valid for all `m` inner updates because the delayed encoder and backbone do not change until the last inner step.
4. For each of `m` inner updates, `model.head` scores the (possibly detached) features, `delayed_model.head` scores the delayed features (the delayed head from after the previous inner Polyak), and `DqnObjective` computes the DQN loss.
5. Two `AdamW` optimizers are required. A single optimizer over `model.parameters()` would still apply Adam moments and weight decay to the encoder and backbone on the head-only backwards. The head optimizer steps every inner update. The feature optimizer (encoder + backbone) steps only on the last inner update, when features stay on the tape.
6. After every head step, `polyak.update` interpolates the delayed head (`tau_heads=POLYAK_TAU_HEADS`). Encoder and backbone `tau` are `0` on inner steps (skipped; no delayed-parameter writes) and `POLYAK_TAU_ENCODER` / `POLYAK_TAU_BACKBONE` on the last inner step. `1` copies the online weights. Every interpolated parameter is fp32, so a small `tau` is never rounded away.

`DqnObjective` interprets `episode_done` and `task_done` (each `0`/`1`/`2`) through separate discount factors. The bootstrap is multiplied by the episode gamma, then by the task gamma (`1.0` when `task_done` is `0`). When a task ends both fire, so a task gamma of `0.0` zeros the whole term.


In [ ]:
head_optimizer = AdamW(
    model.heads.parameters(),
    lr=1e-05,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
feature_optimizer = AdamW(
    list(model.encoder.parameters()) + list(model.backbone.parameters()),
    lr=1e-05,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
delayed_model = model.delayed_copy()
polyak = Polyak(model, delayed_model)
objective = DqnObjective(
    gamma_step=1.0,
    gamma_episode_terminal=1.0,
    gamma_episode_truncated=1.0,
    gamma_task_terminal=0.0,
    gamma_task_truncated=0.0,
    grouping_field="task_index",
)

def run_train(*, model: Model, delayed_model: Model, polyak: Polyak, head_optimizer: AdamW, feature_optimizer: AdamW, objective: DqnObjective, loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float]]:
    """Run ``num_steps`` composite updates on batches from ``loader``.

    Each composite update is ``HEAD_UPDATES`` head steps and one encoder/backbone step.
    """
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        features = model.features(inputs)
        with torch.no_grad():
            delayed_features = delayed_model.features(inputs)
        objective_data = objective_data.to(device)
        for i in range(HEAD_UPDATES):
            last = i == HEAD_UPDATES - 1
            h = features if last else features.detach()
            preds = model.head(h=h)
            with torch.no_grad():
                delayed_preds = delayed_model.head(h=delayed_features)
            loss, metrics = objective(objective_data, preds, delayed_preds)
            head_optimizer.zero_grad()
            if last:
                feature_optimizer.zero_grad()
            loss.backward()
            head_optimizer.step()
            if last:
                feature_optimizer.step()
            tau_heads = POLYAK_TAU_HEADS
            tau_encoder = POLYAK_TAU_ENCODER if last else 0.0
            tau_backbone = POLYAK_TAU_BACKBONE if last else 0.0
            polyak.update(
                tau_heads=tau_heads,
                tau_encoder=tau_encoder,
                tau_backbone=tau_backbone,
            )
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the checkpoint later in `09_inference.ipynb`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(
        model=model,
        delayed_model=delayed_model,
        polyak=polyak,
        head_optimizer=head_optimizer,
        feature_optimizer=feature_optimizer,
        objective=objective,
        loader=loader,
        num_steps=TRAIN_STEPS,
    )
    print(f"cycle={cycle} train  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}")
loader.close()


## Push To The Hub

`push_model_to_hub` uploads the model to `MODEL_ID` and the tokenizer packing spec to a different repo (`TOKENIZER_ID`). Later, `load_model` reconstructs the `Model` and `load_tokenizer` on the tokenizer repo reloads the packing spec — formats, skips, and `head_output` cannot be recovered from the embedder alone.


In [ ]:
model.eval().to("cpu")
model_url, tokenizer_url = push_model_to_hub(model=model, tokenizer=tokenizer, repo_id=MODEL_ID, tokenizer_repo_id=TOKENIZER_ID, private=False, clear=True)
print(f"Pushed model to {model_url}\nPushed tokenizer to {tokenizer_url}")
